# 02 — Results from W&B

Pulls all runs of the project with `wandb.Api()`. Nothing here is hard-coded: run it after the sweeps, the seed runs and `evaluate.py`.

Columns: `best_val_loss` / `best_val_acc` are the validation metrics of the kept (best-val_loss) checkpoint; `test_acc` exists only for runs evaluated with `src/evaluate.py`.

In [ ]:
import os
import wandb
import pandas as pd
import matplotlib.pyplot as plt

ENTITY = os.environ.get('WANDB_ENTITY')  # or put your entity here
PROJECT = os.environ.get('WANDB_PROJECT', 'nlp-hs26-project1')
api = wandb.Api()
ENTITY = ENTITY or api.default_entity

In [ ]:
rows = []
for run in api.runs(f'{ENTITY}/{PROJECT}'):
    if run.state != 'finished':
        continue
    cfg, s = run.config, run.summary
    rows.append({
        'name': run.name, 'id': run.id, 'group': run.group, 'sweep': run.sweep.id if run.sweep else None,
        'lr': cfg.get('lr'), 'hidden_dim': cfg.get('hidden_dim'), 'dropout': cfg.get('dropout'), 'seed': cfg.get('seed'),
        'best_epoch': s.get('best_epoch'), 'best_val_loss': s.get('best_val_loss'), 'best_val_acc': s.get('best_val_acc'),
        'test_acc': s.get('test_acc'), 'test_loss': s.get('test_loss'), 'checkpoint': s.get('best_checkpoint'),
    })
df = pd.DataFrame(rows)
print(len(df), 'finished runs')
df.head()

## Best run per group (selected by validation loss)

In [ ]:
best = df.dropna(subset=['best_val_loss']).sort_values('best_val_loss').groupby('group').head(1)
best[['group', 'name', 'lr', 'hidden_dim', 'dropout', 'best_epoch', 'best_val_loss', 'best_val_acc', 'test_acc', 'checkpoint']]

## Full sweep tables

In [ ]:
for group, g in df[df['sweep'].notna()].groupby('group'):
    print(group)
    display(g.sort_values('best_val_loss')[['name', 'lr', 'hidden_dim', 'dropout', 'best_epoch', 'best_val_loss', 'best_val_acc']])

## Seed variance of the final configurations (groups `final-*`)

In [ ]:
final = df[df['group'].fillna('').str.startswith('final-')]
final.groupby('group')[['best_val_acc', 'test_acc']].agg(['mean', 'std', 'count'])

## Validation accuracy vs. learning rate

One panel per architecture (sweep runs only). Colour = hidden size (fixed order), line style = dropout. All panels share the y-axis so architectures are comparable.

In [ ]:
SERIES = ['#2a78d6', '#eb6834', '#1baf7a']  # fixed categorical order (slots 1-3)
STYLE = {0.0: '-', 0.3: '--'}
sweeps = df[df['sweep'].notna()]
groups = sorted(sweeps['group'].unique())
fig, axes = plt.subplots(1, len(groups), figsize=(5.5 * len(groups), 4), sharey=True, squeeze=False)
for ax, group in zip(axes[0], groups):
    g = sweeps[sweeps['group'] == group]
    for color, h in zip(SERIES, sorted(g['hidden_dim'].unique())):
        for do, ls in STYLE.items():
            line = g[(g['hidden_dim'] == h) & (g['dropout'] == do)].sort_values('lr')
            if len(line):
                ax.plot(line['lr'], line['best_val_acc'], ls, color=color, lw=2, marker='o', ms=8,
                        label=f'h={h}, dropout={do}')
    ax.set_xscale('log')
    ax.set_title(group)
    ax.set_xlabel('learning rate')
    ax.grid(alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, fontsize=8)
axes[0][0].set_ylabel('validation accuracy (best checkpoint)')
plt.tight_layout()
plt.show()